In [ ]:
import os
import csv
import re
import pandas as pd
import numpy as np
import openpyxl

# ==========================================
# 0. БАЗОВЫЕ НАСТРОЙКИ
# ==========================================
pd.set_option('display.max_rows', None)
file_path = 'new_base.xlsx' 

# ==========================================
# 1. ФУНКЦИИ-ПОМОЩНИКИ
# ==========================================
def get_part_type(name):
    name_lower = name.lower()
    if 'шаров' in name_lower: return 'Шаровая опора'
    if 'наконечн' in name_lower: return 'Рулевой наконечник'
    if 'стаб' in name_lower:
        if 'передн' in name_lower: return 'Стойка стабилизатора передняя'
        return 'Стойка стабилизатора'
    if 'сошк' in name_lower: return 'Сошка рулевая'
    if 'маятник' in name_lower: return 'Рычаг маятниковый'
    if 'рычаг подвески' in name_lower: return 'Рычаг подвески'
    if 'поперечн' in name_lower: return 'Тяга рулевая поперечная'
    if 'центральн' in name_lower: return 'Тяга рулевая центральная'
    if 'тяга' in name_lower: return 'Тяга рулевая'
    return None

def get_555_article(name):
    """Ищет артикул 555 (например: SE-3903, SB-N162, SLH-520L)."""
    
    # Расширили Группу 1: добавили SL[A-Z] для перехвата SLH, SLN, SLK и т.д.
    match = re.search(r'(?<![A-Za-z0-9-])(S[ABCEILPRT]|SRT|SL[A-Z])\s?[-]?\s?([A-Z]?\d+[A-Z0-9]*(?:\s?[LR])?)(?![A-Za-z0-9-])', name)
    
    if match:
        prefix = match.group(1).replace(' ', '')
        suffix = match.group(2).replace(' ', '')
        
        # Надежно склеиваем префикс и суффикс через дефис
        return f"{prefix}-{suffix}"
        
    return None

def get_ctr_article(name):
    match = re.search(r'(?<![A-Za-z0-9-])([CС][A-Z]+-\d+[A-Z]?)(?![A-Za-z0-9-])', name)
    if match: return match.group(1).replace('С', 'C')
    return None

def get_lynx_article(name):
    match = re.search(r'(?<![A-Za-z0-9-])(C\d{4}(?:[LR]|LR)?|SS-\d{4})(?![A-Za-z0-9-])', name)
    if match: return match.group(1)
    return None

def get_sufix_article(name):
    match = re.search(r'(?<![A-Za-z0-9-])(S[A-Z])-?(\d{4})(?![A-Za-z0-9-])', name)
    if match: return f"{match.group(1)}-{match.group(2)}"
    return None

def get_tiguar_article(name):
    match = re.search(r'(?<![A-Za-z0-9-])(TG-[A-Za-z0-9-]+)(?![A-Za-z0-9-])', name, re.IGNORECASE)
    if match: return match.group(1).upper()
    return None

def get_specific_brand_article(name, brand):
    if not brand: return None
    b = brand.upper()
    
    if b == 'KYB': m = re.search(r'(?<![A-Za-z0-9-])(K[A-Z]{2,3}\d+)(?![A-Za-z0-9-])', name)
    elif b == 'AVANTECH': m = re.search(r'(?<![A-Za-z0-9-])(A[A-Z]{2}\d{4}[A-Z]?)(?![A-Za-z0-9-])', name)
    elif b == 'JIKIU': m = re.search(r'(?<![A-Za-z0-9-])([A-Z]{2,3}\d{4,5}[A-Z]?)(?![A-Za-z0-9-])', name)
# Добавили [A-Z]? перед закрывающей скобкой, чтобы ловить R или L
    elif b == 'JETT': m = re.search(r'(?<![A-Za-z0-9-])(V\d{2}-\d{3}[A-Z]?)(?![A-Za-z0-9-])', name)
    elif b == 'MILES': m = re.search(r'(?<![A-Za-z0-9-])(D[A-Z]\d{5})(?![A-Za-z0-9-])', name)
    elif b == 'APLUS': m = re.search(r'(?<![A-Za-z0-9-])(\d+AP)(?![A-Za-z0-9-])', name)
    elif b == 'MASUMA': m = re.search(r'(?<![A-Za-z0-9-])(ML-\d+)(?![A-Za-z0-9-])', name)
    elif b == 'FEBEST': m = re.search(r'(?<![A-Za-z0-9-])(\d{4}-[A-Z0-9]+)(?![A-Za-z0-9-])', name)
    elif b == 'BAIKOR': m = re.search(r'(?<![A-Za-z0-9-])(BK[A-Z]+\d+)(?![A-Za-z0-9-])', name)
    elif b == 'GMB': m = re.search(r'(?<![A-Za-z0-9-])(\d{4}-\d{4}|08\d+)(?![A-Za-z0-9-])', name)
    elif b == 'NIPPARTS': m = re.search(r'(?<![A-Za-z0-9-])(J\d{7})(?![A-Za-z0-9-])', name)
    elif b == 'TATSUMI': m = re.search(r'(?<![A-Za-z0-9-])(T[A-Z]{2}\d+)(?![A-Za-z0-9-])', name)
    elif b == 'PARTRA': m = re.search(r'(?<![A-Za-z0-9-])(AJ\d+)(?![A-Za-z0-9-])', name)
    elif b == 'BLUE PRINT': m = re.search(r'(?<![A-Za-z0-9-])(ADT\d+)(?![A-Za-z0-9-])', name)
    elif b in ['SIDEM', 'FEBI']: m = re.search(r'(?<![A-Za-z0-9-])(\d{5})(?![A-Za-z0-9-])', name)
    elif b == 'SAT': m = re.search(r'(?<![A-Za-z0-9-])(ST-[A-Z0-9-]+)(?![A-Za-z0-9-])', name)
    elif b == 'BM':
        m = re.search(r'(?<![A-Za-z0-9-])(TR\s?\d+)(?![A-Za-z0-9-])', name)
        if m: return m.group(1).replace(' ', '')
    elif b == 'AISK': m = re.search(r'(?<![A-Za-z0-9-])(SL\d+)(?![A-Za-z0-9-])', name)
    elif b == 'QSTEN': m = re.search(r'(?<![A-Za-z0-9-])(10660|A01[A-Z0-9]+)(?![A-Za-z0-9-])', name)
    else: return None

    if 'm' in locals() and m: return m.group(1)
    return None

def get_oem_number(name, found_articles):
    match_oem_strict = re.search(r'(?<!\d)(\d{5}-[A-Za-z0-9-]+)', name)
    if match_oem_strict: return match_oem_strict.group(1).strip('-')

    tokens = re.findall(r'[A-Za-z0-9-]+', name)
# Просто добавили 'V' в конец списка, чтобы OEM-поиск сразу отбрасывал такие токены
    bad_prefixes = ('CB', 'SB', 'CE', 'SE', 'CL', 'SL', 'CR', 'SR', 'TY-', '0521', '0123', '2123', 'V')
    bad_exact_words = {'555', 'SUFIX', 'LYNX', 'LYNXAUTO', 'LINX', 'CHASE'}
    
    for token in tokens:
        token_upper = token.upper()
        if token in found_articles or '0123' in token or token_upper in bad_exact_words: continue
        if any(token_upper.startswith(prefix) for prefix in bad_prefixes): continue
        if len(token) > 6 and '-' in token and re.search(r'\d{3}', token): return token
    return None

def get_oem_number_advanced(name, found_articles):
    match_sat = re.search(r'ST-(\d{5}-[A-Za-z0-9-]+)', name)
    if match_sat: return match_sat.group(1)
    
    match_mazda = re.search(r'(?<![A-Za-z0-9-])([A-Z0-9]{4}-\d{2}-\d{3}[A-Z]*)(?![A-Za-z0-9-])', name)
    if match_mazda and match_mazda.group(1) not in found_articles: return match_mazda.group(1)
    
    match_mr = re.search(r'(?<![A-Za-z0-9-])(MR\s?\d{6})(?![A-Za-z0-9-])', name)
    if match_mr and match_mr.group(1) not in found_articles: return match_mr.group(1).replace(' ', '')
    
    return get_oem_number(name, found_articles)

def is_valid(val):
    return pd.notna(val) and str(val).strip() != '' and str(val).strip().lower() != 'none'


# ==========================================
# 2. ИМПОРТ EXCEL И ВОССТАНОВЛЕНИЕ ИЕРАРХИИ
# ==========================================
print("ЭТАП 1: Чтение Excel и восстановление иерархии...")
wb = openpyxl.load_workbook(file_path)
ws = wb.active
levels = [ws.row_dimensions[i].outline_level for i in range(2, ws.max_row + 1)]

df = pd.read_excel(file_path)
df = df[['Код', 'Артикул', 'Наименование', 'Единица']]
df['Уровень_глубины'] = levels[:len(df)] 

current_path = {}
category_list = []

for index, row in df.iterrows():
    lvl = row['Уровень_глубины']
    name = str(row['Наименование']).strip()
    
    if pd.isna(row['Единица']):
        current_path[lvl] = name
        keys_to_delete = [k for k in current_path.keys() if k > lvl]
        for k in keys_to_delete: del current_path[k]
        category_list.append(np.nan) 
    else:
        sorted_levels = sorted(current_path.keys())
        full_category_name = "/".join([current_path[k] for k in sorted_levels])
        category_list.append(full_category_name)

df['Категория'] = category_list
df = df.dropna(subset=['Единица']).copy()
df = df.drop(columns=['Уровень_глубины'])
df['Главная категория'] = df['Категория'].apply(lambda x: str(x).split('/')[0])

# --- НОВЫЙ БЛОК: Изоляция целевых категорий ---
print("ЭТАП 1.5: Отделение целевых категорий от остальной базы...")
def is_target_category(cat):
    cat_upper = str(cat).strip().upper()
    if cat_upper.startswith(('555/', 'CHASE/', 'CTR/', 'LYNX/', 'LYNXAUTO/', 'SUFIX/', 'TI-GUAR/')):
        return True
    # if 'TI-GUAR' in cat_upper or 'TIGUAR' in cat_upper:
    #     return True
    return False

mask = df['Категория'].apply(is_target_category)

df_untouched = df[~mask].copy() # Откладываем в сторону то, что не обрабатываем
df = df[mask].copy()            # Оставляем в df только нужные категории
print(f"Отобрано для обработки: {len(df)} строк. Отложено: {len(df_untouched)} строк.")
# ----------------------------------------------
# ==========================================
# 3. ПЕРВЫЙ ПРОХОД: РАЗБОР ПО КАТЕГОРИЯМ
# ==========================================
print("ЭТАП 2: Основная логика (Первый проход)...")
columns = ['Артикул_SUFIX', 'Артикул_LYNX', 'Артикул_555', 'Артикул_CTR', 'Артикул_Ti-Guar', 'Бренд', 'Тип_детали', 'Оригинальный_номер']
for col in columns: df[col] = None

items_to_move = []
blacklist = [
    "MR179480S1 шаровая опора SWAG COLT/GALANT/LANCER 555",
    "0521-DY3LH D350-32-290A рулевой наконечник Demio DY3/5 LH CHASE",
    "SC 4680 тяга рулевая  поперечная 555",
    "Стойка стабилизатора переднего NISSAN VANETTE/BONGO  OEM"
]
rename_dict = {
    "C42292LR": "C4292LR",
    "LSE-": "SE-",
    "--": "-",
    "0123-EXZ10F 48819-10010 rbi": "0123-EXZ10F стойка стабилизатора 48819-10010 rbi",
    "A01RE 10660 рулевая тяга NZE12 QSTEN": "A01RE10660 рулевая тяга NZE12 QSTEN",
    "CRHO-51/SR-HO40 тяга рулевая 555": "CRHO-51/SR-H040 тяга рулевая 555"
}

for row in df.index:
    name = str(df.loc[row, 'Наименование'])
    cat = str(df.loc[row, 'Категория'])
    
    if name in blacklist:
        df.drop(row, inplace=True)
        continue
        
    for old, new in rename_dict.items():
        if old in name:
            name = name.replace(old, new)
            df.loc[row, 'Наименование'] = name
            
    brand = None
    art_555, art_ctr, art_lynx, art_sufix, art_tiguar = None, None, None, None, None
    
    if cat.startswith('555/') or cat.startswith('CHASE/') or cat.startswith('CTR/'):
        brand = cat.split('/')[0]
        if brand not in name.upper():
            items_to_move.append(df.loc[row].to_dict())
            df.drop(row, inplace=True)
            continue
        art_555, art_ctr = get_555_article(name), get_ctr_article(name)
        
    elif cat.startswith('LYNX/') or cat.startswith('LYNXauto/'):
        if 'LYNX' not in name.upper() and 'LINX' not in name.upper():
            items_to_move.append(df.loc[row].to_dict())
            df.drop(row, inplace=True)
            continue
        brand = 'LYNXauto'
        art_lynx, art_ctr, art_555 = get_lynx_article(name), get_ctr_article(name), get_555_article(name)
        
    elif cat.startswith('SUFIX/'):
        if 'SUFIX' not in name.upper():
            items_to_move.append(df.loc[row].to_dict())
            df.drop(row, inplace=True)
            continue
        brand = 'SUFIX'
        art_sufix, art_ctr, art_lynx = get_sufix_article(name), get_ctr_article(name), get_lynx_article(name)
        
    elif 'TI-GUAR' in cat.upper() or 'TIGUAR' in cat.upper():
        if not re.search(r'TI[-•\s]*GUAR', name, re.IGNORECASE):
            items_to_move.append(df.loc[row].to_dict())
            df.drop(row, inplace=True)
            continue
        brand = 'Ti-Guar'
        art_tiguar, art_ctr, art_555 = get_tiguar_article(name), get_ctr_article(name), get_555_article(name)
    else:
        continue

    found_articles = [art for art in (art_555, art_ctr, art_lynx, art_sufix, art_tiguar) if art]
    part_type = get_part_type(name)
    oem_number = get_oem_number(name, found_articles)
 
    if brand == 'Ti-Guar':
        if not oem_number:
            match_7_digit = re.search(r'(?<![A-Za-z0-9-])(\d{7})(?![A-Za-z0-9-])', name)
            if match_7_digit: oem_number = match_7_digit.group(1)
        if not art_tiguar and oem_number:
            art_tiguar = 'TG-' + oem_number

    df.loc[row, 'Бренд'], df.loc[row, 'Тип_детали'], df.loc[row, 'Оригинальный_номер'] = brand, part_type, oem_number
    if art_555: df.loc[row, 'Артикул_555'] = art_555
    if art_ctr: df.loc[row, 'Артикул_CTR'] = art_ctr
    if art_lynx: df.loc[row, 'Артикул_LYNX'] = art_lynx
    if art_sufix: df.loc[row, 'Артикул_SUFIX'] = art_sufix
    if art_tiguar: df.loc[row, 'Артикул_Ti-Guar'] = art_tiguar

# ==========================================
# 4. ВТОРОЙ ПРОХОД: ОТСТОЙНИК (DF_2)
# ==========================================
print("ЭТАП 3: Обработка отстойника (Второй проход)...")
df_2 = pd.DataFrame(items_to_move) if items_to_move else pd.DataFrame(columns=df.columns)
for col in columns:
    if col not in df_2.columns: df_2[col] = None
    df_2[col] = df_2[col].astype(object)

final_items_to_move = []
for row in df_2.index:
    name, name_upper = str(df_2.loc[row, 'Наименование']), str(df_2.loc[row, 'Наименование']).upper()
    brand = None
    art_555, art_ctr, art_lynx, art_sufix, art_tiguar = None, None, None, None, None
    
    if re.search(r'TI[-•\s]*GUAR', name, re.IGNORECASE): brand = 'Ti-Guar'
    elif 'LYNX' in name_upper or 'LINX' in name_upper: brand = 'LYNXauto'
    elif 'SUFIX' in name_upper: brand = 'SUFIX'
    elif 'CHASE' in name_upper: brand = 'CHASE'
    elif '555' in name_upper: brand = '555'
    else:
        final_items_to_move.append(df_2.loc[row].to_dict())
        df_2.drop(row, inplace=True)
        continue
        
    art_ctr = get_ctr_article(name)
    if brand in ['555', 'CHASE']: art_555 = get_555_article(name)
    elif brand == 'LYNXauto': art_lynx, art_555 = get_lynx_article(name), get_555_article(name)
    elif brand == 'SUFIX': art_sufix, art_lynx = get_sufix_article(name), get_lynx_article(name)
    elif brand == 'Ti-Guar': art_tiguar, art_555 = get_tiguar_article(name), get_555_article(name)
        
    found_articles = [art for art in (art_555, art_ctr, art_lynx, art_sufix, art_tiguar) if art]
    part_type = get_part_type(name)
    oem_number = get_oem_number(name, found_articles)

    if not part_type:
        final_items_to_move.append(df_2.loc[row].to_dict())
        df_2.drop(row, inplace=True)
        continue
    
    if brand == 'Ti-Guar':
        if not oem_number:
            match_7_digit = re.search(r'(?<![A-Za-z0-9-])(\d{7})(?![A-Za-z0-9-])', name)
            if match_7_digit: oem_number = match_7_digit.group(1)
        if not art_tiguar and oem_number: art_tiguar = 'TG-' + oem_number

    df_2.loc[row, 'Бренд'], df_2.loc[row, 'Тип_детали'], df_2.loc[row, 'Оригинальный_номер'] = brand, part_type, oem_number
    if art_555: df_2.loc[row, 'Артикул_555'] = art_555
    if art_ctr: df_2.loc[row, 'Артикул_CTR'] = art_ctr
    if art_lynx: df_2.loc[row, 'Артикул_LYNX'] = art_lynx
    if art_sufix: df_2.loc[row, 'Артикул_SUFIX'] = art_sufix
    if art_tiguar: df_2.loc[row, 'Артикул_Ti-Guar'] = art_tiguar

# ==========================================
# 5. ТРЕТИЙ ПРОХОД: ОТСТОЙНИК ОТСТОЙНИКОВ
# ==========================================
print("ЭТАП 4: Разбор финального отстойника на бренды...")
df_final_moved = pd.DataFrame(final_items_to_move) if final_items_to_move else pd.DataFrame(columns=df_2.columns)
salvaged_items, swamp_items = [], []

for row in df_final_moved.index:
    name = str(df_final_moved.loc[row, 'Наименование'])
    part_type = get_part_type(name)
    if part_type:
        item_dict = df_final_moved.loc[row].to_dict()
        item_dict['Тип_детали'] = part_type 
        salvaged_items.append(item_dict)
    else:
        swamp_items.append(df_final_moved.loc[row].to_dict())

if swamp_items:
    pd.DataFrame(swamp_items).to_csv('swamp.csv', index=False, sep=';', encoding='utf-8-sig', quoting=csv.QUOTE_ALL)

df_3 = pd.DataFrame(salvaged_items) if salvaged_items else pd.DataFrame(columns=df_2.columns)
brand_names = ['KYB','Avantech','JIKIU','RBI','FEBI','JETT','SAT','Febest','MILES','Aplus',
               'Tenacity','R8','Sidem','Blue Print','GMB','MASUMA','Baikor','Nipparts',
               'PARTRA','Tatsumi','QSTEN','CTR','AISK','BM']

brand_dfs = []
for brand_name in brand_names:
    if df_3.empty: break
    mask = df_3["Наименование"].str.contains(brand_name, case=False, na=False)
    df_brand = df_3[mask].copy()
    if not df_brand.empty:
        df_brand['Бренд'] = brand_name
        brand_dfs.append(df_brand)
    df_3 = df_3[~mask]

if not df_3.empty:
    df_3['Бренд'] = 'OEM' # ИСПРАВЛЕНА ОШИБКА ЗДЕСЬ!
    brand_dfs.append(df_3)

df_branded = pd.concat(brand_dfs, ignore_index=True) if brand_dfs else pd.DataFrame(columns=df_2.columns)

# ==========================================
# 6. ЧЕТВЕРТЫЙ ПРОХОД: ПАРСИНГ СПЕЦ. БРЕНДОВ
# ==========================================
print("ЭТАП 5: Детальный парсинг специфичных брендов...")
new_cols = ['Артикул_Бренда', 'Артикул_555', 'Артикул_CTR', 'Артикул_LYNX', 'Тип_детали', 'Оригинальный_номер']
for col in new_cols:
    if col not in df_branded.columns: df_branded[col] = None
    df_branded[col] = df_branded[col].astype(object)

for row in df_branded.index:
    name, brand = str(df_branded.loc[row, 'Наименование']), str(df_branded.loc[row, 'Бренд'])
    
    art_555, art_ctr, art_lynx = get_555_article(name), get_ctr_article(name), get_lynx_article(name)
    art_brand_specific = get_specific_brand_article(name, brand)
    
    found_articles = [art for art in (art_555, art_ctr, art_lynx, art_brand_specific) if art]
    part_type = get_part_type(name)
    oem_number = get_oem_number_advanced(name, found_articles)
    
    df_branded.loc[row, 'Тип_детали'], df_branded.loc[row, 'Оригинальный_номер'] = part_type, oem_number
    if art_555: df_branded.loc[row, 'Артикул_555'] = art_555
    if art_ctr: df_branded.loc[row, 'Артикул_CTR'] = art_ctr
    if art_lynx: df_branded.loc[row, 'Артикул_LYNX'] = art_lynx
    if art_brand_specific: df_branded.loc[row, 'Артикул_Бренда'] = art_brand_specific

# ==========================================
# 7. ФИНАЛЬНОЕ СЛИЯНИЕ И СЖАТИЕ
# ==========================================
print("ЭТАП 6: Сборка всех данных и сжатие (Waterfall Compression)...")
df_final = pd.concat([df_branded, df_2, df], ignore_index=True)

def get_main_article(row):
    brand = str(row.get('Бренд')).upper()
    
    # 1. Пытаемся взять "законный" артикул бренда
    if is_valid(row.get('Артикул_Бренда')): return row['Артикул_Бренда']
    if 'CHASE' in brand and is_valid(row.get('Артикул_555')): return row['Артикул_555']
    if '555' in brand and is_valid(row.get('Артикул_555')): return row['Артикул_555']
    if 'CTR' in brand and is_valid(row.get('Артикул_CTR')): return row['Артикул_CTR']
    if 'LYNX' in brand and is_valid(row.get('Артикул_LYNX')): return row['Артикул_LYNX']
    if 'SUFIX' in brand and is_valid(row.get('Артикул_SUFIX')): return row['Артикул_SUFIX']
    if ('TI-GUAR' in brand or 'TIGUAR' in brand) and is_valid(row.get('Артикул_Ti-Guar')): return row['Артикул_Ti-Guar']
        
    # 2. СПАСАТЕЛЬНЫЙ КРУГ (Твоя идея): 
    # Если родного артикула нет, "повышаем" 555, а если его нет — LYNX
    if is_valid(row.get('Артикул_555')): return row['Артикул_555']
    if is_valid(row.get('Артикул_LYNX')): return row['Артикул_LYNX']
    
    return None

def get_cross_article(row):
    # Узнаем, какой номер мы уже забрали в главную колонку
    main_art = get_main_article(row)
    
    # Собираем всех кандидатов в кросс-номера в порядке приоритета
    candidates = [
        row.get('Артикул_CTR'),
        row.get('Артикул_555'),
        row.get('Артикул_LYNX'),
        row.get('Артикул_SUFIX'),
        row.get('Артикул_Ti-Guar')
    ]
    
    # Берем первый существующий номер, который ЕЩЕ НЕ ИСПОЛЬЗОВАН как основной
    for cand in candidates:
        if is_valid(cand) and cand != main_art:
            return cand
            
    return None

df_final['Финальный_Артикул_Бренда'] = df_final.apply(get_main_article, axis=1)
df_final['Артикул_Кросс'] = df_final.apply(get_cross_article, axis=1)

columns_to_drop = ['Артикул_555', 'Артикул_CTR', 'Артикул_LYNX', 'Артикул_SUFIX', 'Артикул_Ti-Guar', 'Артикул_Бренда']
df_final.drop(columns=[col for col in columns_to_drop if col in df_final.columns], inplace=True)
df_final.rename(columns={'Финальный_Артикул_Бренда': 'Артикул_Бренда'}, inplace=True)

# Сортировка столбцов
front_cols = ['Бренд', 'Тип_детали', 'Артикул_Бренда', 'Артикул_Кросс', 'Оригинальный_номер', 'Наименование', 'Категория']
final_columns = [col for col in front_cols if col in df_final.columns]
for col in df_final.columns:
    if col not in final_columns: final_columns.append(col)
df_final = df_final[final_columns]

# Сохранение финального результата
df_final.to_csv('ultimate_parts_compressed.csv', index=False, sep=';', encoding='utf-8-sig', quoting=csv.QUOTE_ALL)
print("Готово! Таблица сжата и сохранена.")

ЭТАП 1: Чтение Excel и восстановление иерархии...
ЭТАП 1.5: Отделение целевых категорий от остальной базы...
Отобрано для обработки: 1731 строк. Отложено: 17866 строк.
ЭТАП 2: Основная логика (Первый проход)...
ЭТАП 3: Обработка отстойника (Второй проход)...
ЭТАП 4: Разбор финального отстойника на бренды...
ЭТАП 5: Детальный парсинг специфичных брендов...
ЭТАП 6: Сборка всех данных и сжатие (Waterfall Compression)...
Готово! Таблица сжата и сохранена.
